In [8]:
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import (balanced_accuracy_score, accuracy_score, precision_score, 
                           log_loss, recall_score, f1_score, roc_auc_score, brier_score_loss)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression
import pylab as pl
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)

In [9]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [10]:
file_name = "final_df.csv"
df = pd.read_csv(file_name)
df.dropna(inplace=True)
df['PDI'] = df['PDI'].astype(int)
df['PDI_complete'] = df['PDI_complete'].astype(int)

In [11]:
def stepwise_selection(X, y, 
                       initial_list=[], 
                       threshold_in=0.05, 
                       threshold_out = 0.1, 
                       verbose=True):
    included = list(initial_list)
    while True:
        changed=False
        # forward step
        excluded = list(set(X.columns)-set(included))
        new_pval = pd.Series(index=excluded)
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(X[included+[new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        best_pval = new_pval.min()
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed=True
            if verbose:
                print('Add  {:30} with p-value {:.4}'.format(best_feature, best_pval))

        # backward step
        model = sm.OLS(y, sm.add_constant(X[included])).fit()
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max()
        if worst_pval > threshold_out:
            changed=True
            worst_feature = pvalues.argmax()
            included.remove(worst_feature)
            if verbose:
                print('Drop {:30} with p-value {:.4}'.format(worst_feature, worst_pval))
        if not changed:
            break
    return included

def find_best_threshold(y_true, y_proba):
    thresholds = np.linspace(0, 1, 101)
    best_thresh = 0
    best_bal_acc = 0
    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        score = balanced_accuracy_score(y_true, preds)
        if score > best_bal_acc:
            best_thresh = t
            best_bal_acc = score
    return best_thresh, best_bal_acc

def compute_detailed_metrics(y_true, y_proba, threshold, model_name):
    y_pred = (y_proba >= threshold).astype(int)
    
    # Calculate confusion matrix components
    tn = np.sum((y_pred == 0) & (y_true == 0))
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    
    # Calculate metrics
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    logloss = log_loss(y_true, y_proba)
    auc_score = roc_auc_score(y_true, y_proba)
    brier_score = brier_score_loss(y_true, y_proba)
    
    return {
        'Model': model_name,
        'Threshold': f"{threshold:.3f}",
        'Balanced_Acc': f"{balanced_acc:.4f}",
        'TNR': f"{tnr:.4f}",
        'Recall': f"{recall:.4f}",
        'F1': f"{f1:.4f}",
        'AUC': f"{auc_score:.4f}",
        'LogLoss': f"{logloss:.4f}",
        'Brier_Score': f"{brier_score:.4f}"
    }

## 1. Without tracking data

In [12]:
# Model 1: Without tracking data
df_without_tracking = df[["R","shooting_angle","assist_type_through_pass","assist_type_long_pass","shot_bodyPart_head_or_other","direct_free_kick","shot_isGoal"]]

y1 = df_without_tracking["shot_isGoal"]
X1 = df_without_tracking.drop(columns="shot_isGoal")

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=80, stratify=y1)

print("Stepwise selection for Model 1 (without tracking)...")
result1 = stepwise_selection(X1_train, y1_train)
print('Selected features:', result1)

# Prepare data for statsmodels
X1_train_stepwise = X1_train[result1]
X1_test_stepwise = X1_test[result1]
X1_train_stepwise = sm.add_constant(X1_train_stepwise)
X1_test_stepwise = sm.add_constant(X1_test_stepwise)

# Fit logistic regression
logit_model1 = sm.Logit(y1_train, X1_train_stepwise)
logit_result1 = logit_model1.fit_regularized()

y1_test_proba = logit_result1.predict(X1_test_stepwise)
best_thresh1, best_bal_acc1 = find_best_threshold(y1_test, y1_test_proba)

print(f"Model 1 - Best threshold: {best_thresh1:.3f}, Balanced Accuracy: {best_bal_acc1:.4f}")


Stepwise selection for Model 1 (without tracking)...
Add  shooting_angle                 with p-value 1.423e-45
Add  assist_type_through_pass       with p-value 4.234e-05
Add  R                              with p-value 9.338e-06
Add  shot_bodyPart_head_or_other    with p-value 0.0001835
Add  direct_free_kick               with p-value 0.0329
Selected features: ['shooting_angle', 'assist_type_through_pass', 'R', 'shot_bodyPart_head_or_other', 'direct_free_kick']
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.33618969239766683
            Iterations: 43
            Function evaluations: 49
            Gradient evaluations: 43
Model 1 - Best threshold: 0.130, Balanced Accuracy: 0.6646


## 2. With tracking data

In [13]:
df_with_tracking = df[["R","theta","shooting_angle","assist_type_through_pass","assist_type_long_pass",
                      "shot_bodyPart_head_or_other","direct_free_kick","PDI","PDI_complete","gk_line_offset","shot_isGoal"]]

y2 = df_with_tracking["shot_isGoal"]
X2 = df_with_tracking.drop(columns="shot_isGoal")

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=80, stratify=y2)

print("Stepwise selection for Model 2 (with tracking)...")
result2 = stepwise_selection(X2_train, y2_train)
print('Selected features:', result2)

# Prepare data for statsmodels
X2_train_stepwise = X2_train[result2]
X2_test_stepwise = X2_test[result2]
X2_train_stepwise = sm.add_constant(X2_train_stepwise)
X2_test_stepwise = sm.add_constant(X2_test_stepwise)

# Fit logistic regression
logit_model2 = sm.Logit(y2_train, X2_train_stepwise)
logit_result2 = logit_model2.fit_regularized()

y2_test_proba = logit_result2.predict(X2_test_stepwise)
best_thresh2, best_bal_acc2 = find_best_threshold(y2_test, y2_test_proba)

print(f"Model 2 - Best threshold: {best_thresh2:.3f}, Balanced Accuracy: {best_bal_acc2:.4f}")

Stepwise selection for Model 2 (with tracking)...
Add  shooting_angle                 with p-value 1.423e-45
Add  gk_line_offset                 with p-value 1.733e-09
Add  assist_type_through_pass       with p-value 9.493e-05
Add  R                              with p-value 0.0001027
Add  theta                          with p-value 1.36e-05
Add  shot_bodyPart_head_or_other    with p-value 2.872e-06
Add  direct_free_kick               with p-value 0.0241
Add  PDI_complete                   with p-value 0.02804
Selected features: ['shooting_angle', 'gk_line_offset', 'assist_type_through_pass', 'R', 'theta', 'shot_bodyPart_head_or_other', 'direct_free_kick', 'PDI_complete']
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.33082375301702605
            Iterations: 50
            Function evaluations: 56
            Gradient evaluations: 50
Model 2 - Best threshold: 0.090, Balanced Accuracy: 0.6544


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


## MODEL COMPARISON

In [14]:
# Detailed metrics evaluation
detailed_results = []
detailed_results.append(compute_detailed_metrics(y1_test, y1_test_proba, best_thresh1, "NoTracking"))
detailed_results.append(compute_detailed_metrics(y2_test, y2_test_proba, best_thresh2, "WithTracking"))

detailed_df = pd.DataFrame(detailed_results)
print("\nDetailed Metrics (Balanced Accuracy Optimal Thresholds):")
print("=" * 90)
detailed_df



Detailed Metrics (Balanced Accuracy Optimal Thresholds):


,Model,Threshold,Balanced_Acc,TNR,Recall,F1,AUC,LogLoss,Brier_Score
0,NoTracking,0.130,0.6646,0.8335,0.4957,0.3591,0.6803,0.3346,0.0957
1,WithTracking,0.090,0.6544,0.5737,0.7350,0.2955,0.7075,0.3330,0.0952
